In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [2]:
root_directory = Path().resolve().parent
DATA_FOLDER = root_directory / 'for_students'
pd.set_option('display.max_columns', None)

In [3]:
accounts = pd.read_csv(DATA_FOLDER / 'account.csv', encoding='latin1')
concerts_2014_2015 = pd.read_csv(DATA_FOLDER / 'concerts_2014-15.csv')
concerts = pd.read_csv(DATA_FOLDER / 'concerts.csv')
subscriptions = pd.read_csv(DATA_FOLDER / 'subscriptions.csv')
tickets_all = pd.read_csv(DATA_FOLDER / 'tickets_all.csv')
zipcodes = pd.read_csv(DATA_FOLDER / 'zipcodes.csv')
train = pd.read_csv(DATA_FOLDER / 'train.csv')

In [4]:
accounts.head()

,account.id,shipping.zip.code,billing.zip.code,shipping.city,billing.city,relationship,amount.donated.2013,amount.donated.lifetime,no.donations.lifetime,first.donated
0,001i000000LhSrQ,94102,94102,San Francisco,San Francisco,NaN,0.0,0.0,0,NaN
1,001i000000LhyPD,NaN,94303,NaN,Palo Alto,NaN,0.0,500.0,5,1987-07-04 00:00:00
2,001i000000LhyPE,NaN,94597,NaN,Walnut Creek,NaN,0.0,783.0,7,2008-06-02 00:00:00
3,001i000000LhyPF,NaN,94610,NaN,Oakland,NaN,1500.0,28435.0,28,1986-07-04 00:00:00
4,001i000000LhyPG,NaN,94024,NaN,Los Altos,NaN,300.0,4969.0,16,1997-04-07 00:00:00


## Cleanliness Checks

In [5]:
files = {'accounts': accounts, 'concerts_2014_2015': concerts_2014_2015, 'concerts': concerts, 'subscriptions': subscriptions, 'tickets_all': tickets_all, 'zipcodes': zipcodes, 'train': train}

In [6]:
for name, file in files.items():
    print(f"File: {name}")
    print(f"Shape: {file.shape}")
    print(f"Data Types:\n{file.dtypes}")
    print(f"Missing Values:\n{file.isnull().sum()}")
    print("---------------")

File: accounts
Shape: (19833, 10)
Data Types:
account.id                     str
shipping.zip.code              str
billing.zip.code               str
shipping.city                  str
billing.city                   str
relationship                   str
amount.donated.2013        float64
amount.donated.lifetime    float64
no.donations.lifetime        int64
first.donated                  str
dtype: object
Missing Values:
account.id                     0
shipping.zip.code          19569
billing.zip.code            2955
shipping.city              19561
billing.city                2218
relationship               19172
amount.donated.2013            0
amount.donated.lifetime        0
no.donations.lifetime          0
first.donated              14298
dtype: int64
---------------
File: concerts_2014_2015
Shape: (6, 5)
Data Types:
season            str
concert.name      str
set             int64
who               str
what              str
dtype: object
Missing Values:
season          0
concer

In [7]:
# aggregate subscriptions
subscriptions['multiple.subs'] = subscriptions['multiple.subs'].map({'no': 0, 'yes': 1})
subscriptions['season_start_year'] = subscriptions['season'].str[:4].astype(int)

subscriptions_agg = subscriptions.groupby('account.id', as_index=False).agg(
    season_count=('season', 'count'),
    earliest_season_start_year=('season_start_year', 'min'),
    latest_season_start_year=('season_start_year', 'max'),
    mean_price_level=('price.level', 'mean'),
    mean_subscription_tier=('subscription_tier', 'mean'),
    mean_multiple_subs=('multiple.subs', 'mean'),
    mean_no_seats=('no.seats', 'mean'),
)

subscriptions_agg

,account.id,season_count,earliest_season_start_year,latest_season_start_year,mean_price_level,mean_subscription_tier,mean_multiple_subs,mean_no_seats
0,001i000000LhyPD,12,1993,2013,2.416667,1.833333,0.000000,2.000000
1,001i000000LhyPE,6,2008,2013,2.500000,1.250000,0.166667,2.166667
2,001i000000LhyPF,19,1994,2013,1.187500,1.842105,0.000000,1.842105
3,001i000000LhyPG,16,1996,2013,1.071429,1.812500,0.000000,1.875000
4,001i000000LhyPH,11,2003,2013,1.000000,1.818182,0.000000,1.818182
...,...,...,...,...,...,...,...,...
6262,001i000000V23aW,1,2013,2013,3.000000,1.000000,0.000000,2.000000
6263,001i000000V24lZ,1,2013,2013,2.000000,1.000000,0.000000,2.000000
6264,001i000000V24u1,1,2013,2013,3.000000,0.500000,0.000000,1.000000
6265,001i000000V255R,1,2013,2013,2.000000,1.000000,0.000000,2.000000


In [8]:
tickets_all['price.level'].value_counts()

price.level
4        1042
3         563
2         489
1         435
0          35
Adult      14
Youth      14
GA          6
4.0         2
Name: count, dtype: int64

In [9]:
subscriptions['price.level'].value_counts()

price.level
2.0    11722
1.0     4859
3.0     3827
4.0     3477
0.0     1208
Name: count, dtype: int64

In [10]:
# aggregate tickets
tickets_all['multiple.tickets'] = tickets_all['multiple.tickets'].map({'no': 0, 'yes': 1})
tickets_all['season_start_year'] = tickets_all['season'].str[:4].astype(int)
tickets_all = tickets_all[tickets_all['price.level'].isin(['0', '1', '2', '3', '4', '4.0'])]
tickets_all['price.level'] = tickets_all['price.level'].astype(float)

print(tickets_all.head())

tickets_agg = tickets_all.groupby('account.id', as_index=False).agg(
    season_count=('season', 'count'),
    earliest_season_start_year=('season_start_year', 'min'),
    latest_season_start_year=('season_start_year', 'max'),
    mean_price_level=('price.level', 'mean'),
    mean_multiple_tickets=('multiple.tickets', 'mean'),
    mean_no_seats=('no.seats', 'mean'),
)

tickets_agg

        account.id  price.level  no.seats      marketing.source     season  \
0  001i000000NuOP0          4.0         2                   NaN  2012-2013   
1  001i000000NuOYU          1.0         1                 Other  2013-2014   
2  001i000000NuRS9          4.0         2                   NaN  2011-2012   
3  001i000000NuRPj          3.0         1  Postcard or brochure  2013-2014   
4  001i000000NuRQ1          2.0         2                   NaN  2013-2014   

            location  set  multiple.tickets  season_start_year  
0      San Francisco  4.0                 0               2012  
1    Berkeley Sunday  6.0                 0               2013  
2     Family concert  3.0                 0               2011  
3  Berkeley Saturday  2.0                 0               2013  
4  Berkeley Saturday  1.0                 0               2013  


,account.id,season_count,earliest_season_start_year,latest_season_start_year,mean_price_level,mean_multiple_tickets,mean_no_seats
0,001i000000LhyPj,1,2012,2012,2.000000,0.0,2.000000
1,001i000000LhyQD,1,2010,2010,2.000000,0.0,2.000000
2,001i000000LhyQO,1,2011,2011,2.000000,0.0,1.000000
3,001i000000LhyQj,1,2010,2010,4.000000,0.0,2.000000
4,001i000000LhyQr,3,2010,2012,3.666667,0.0,2.333333
...,...,...,...,...,...,...,...
1878,001i000000q7wB7,1,2013,2013,4.000000,0.0,1.000000
1879,001i000000q7wB8,1,2013,2013,4.000000,0.0,1.000000
1880,001i000000q7wB9,1,2013,2013,2.000000,0.0,2.000000
1881,001i000000q7wBA,1,2013,2013,1.000000,0.0,1.000000


In [11]:
subscriptions_agg.to_csv(DATA_FOLDER / 'subscriptions_agg.csv', index=False)
tickets_agg.to_csv(DATA_FOLDER / 'tickets_agg.csv', index=False)

In [13]:
merged = accounts.merge(
    subscriptions_agg, how='left', on='account.id'
).merge(
    tickets_agg, how='left', on='account.id', suffixes=('_subs', '_tickets')
)

merged.head()

,account.id,shipping.zip.code,billing.zip.code,shipping.city,billing.city,relationship,amount.donated.2013,amount.donated.lifetime,no.donations.lifetime,first.donated,season_count_subs,earliest_season_start_year_subs,latest_season_start_year_subs,mean_price_level_subs,mean_subscription_tier,mean_multiple_subs,mean_no_seats_subs,season_count_tickets,earliest_season_start_year_tickets,latest_season_start_year_tickets,mean_price_level_tickets,mean_multiple_tickets,mean_no_seats_tickets
0,001i000000LhSrQ,94102,94102,San Francisco,San Francisco,NaN,0.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,001i000000LhyPD,NaN,94303,NaN,Palo Alto,NaN,0.0,500.0,5,1987-07-04 00:00:00,12.0,1993.0,2013.0,2.416667,1.833333,0.000000,2.000000,NaN,NaN,NaN,NaN,NaN,NaN
2,001i000000LhyPE,NaN,94597,NaN,Walnut Creek,NaN,0.0,783.0,7,2008-06-02 00:00:00,6.0,2008.0,2013.0,2.500000,1.250000,0.166667,2.166667,NaN,NaN,NaN,NaN,NaN,NaN
3,001i000000LhyPF,NaN,94610,NaN,Oakland,NaN,1500.0,28435.0,28,1986-07-04 00:00:00,19.0,1994.0,2013.0,1.187500,1.842105,0.000000,1.842105,NaN,NaN,NaN,NaN,NaN,NaN
4,001i000000LhyPG,NaN,94024,NaN,Los Altos,NaN,300.0,4969.0,16,1997-04-07 00:00:00,16.0,1996.0,2013.0,1.071429,1.812500,0.000000,1.875000,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
merged['past_subscription'] = merged['season_count_subs'].notnull().astype(int)

In [15]:
merged.head()

,account.id,shipping.zip.code,billing.zip.code,shipping.city,billing.city,relationship,amount.donated.2013,amount.donated.lifetime,no.donations.lifetime,first.donated,season_count_subs,earliest_season_start_year_subs,latest_season_start_year_subs,mean_price_level_subs,mean_subscription_tier,mean_multiple_subs,mean_no_seats_subs,season_count_tickets,earliest_season_start_year_tickets,latest_season_start_year_tickets,mean_price_level_tickets,mean_multiple_tickets,mean_no_seats_tickets,past_subscription
0,001i000000LhSrQ,94102,94102,San Francisco,San Francisco,NaN,0.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,001i000000LhyPD,NaN,94303,NaN,Palo Alto,NaN,0.0,500.0,5,1987-07-04 00:00:00,12.0,1993.0,2013.0,2.416667,1.833333,0.000000,2.000000,NaN,NaN,NaN,NaN,NaN,NaN,1
2,001i000000LhyPE,NaN,94597,NaN,Walnut Creek,NaN,0.0,783.0,7,2008-06-02 00:00:00,6.0,2008.0,2013.0,2.500000,1.250000,0.166667,2.166667,NaN,NaN,NaN,NaN,NaN,NaN,1
3,001i000000LhyPF,NaN,94610,NaN,Oakland,NaN,1500.0,28435.0,28,1986-07-04 00:00:00,19.0,1994.0,2013.0,1.187500,1.842105,0.000000,1.842105,NaN,NaN,NaN,NaN,NaN,NaN,1
4,001i000000LhyPG,NaN,94024,NaN,Los Altos,NaN,300.0,4969.0,16,1997-04-07 00:00:00,16.0,1996.0,2013.0,1.071429,1.812500,0.000000,1.875000,NaN,NaN,NaN,NaN,NaN,NaN,1


In [16]:
merged.to_csv(DATA_FOLDER / 'merged.csv', index=False)